<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a> | <a href="./Access_OLCI_EUMETSAT_Data_Store.ipynb">Accessing Sentinel-3 OLCI data through the EUMETSAT Data Store</a> | <a href="./Access_MSI_CDSE.ipynb">Accessing Sentinel-2 MSI data via the CDSE</a> | <a href="./Access_multisensor_CMEMS_Data_Store.ipynb">Accessing multisensor data through the CMEMS Data Store</a>

**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** Ben Loveday (Innoflair for EUMETSAT), Anna Windle diPaolo (NASA), Carina Poulin (NASA)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following prerequisites:
- **<a href="https://urs.earthdata.nasa.gov/" target="_blank">An Earthdata account</a>** if you are using or plan to use Earthdata to download PACE OCI data.
- **<a href="../submodules/pace-2025/book/presentations/notebooks/earthdata_cloud_access.ipynb" target="_blank">Earthdata Cloud Access</a>** notebook, included in this repository under the pace-2025 submodule.

</div>
<hr>

# Accessing NASA PACE OCI data via Earthdata

### Data used
| Dataset | EarthData collection ID | EarthData collection<br>description |
|:--------------------:|:-----------------------:|:-------------:|
| PACE OCI Level-2 Regional Ocean Biogeochemical Properties Data | PACE_OCI_L2_BGC | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3620139680-OB_CLOUD.html" target="_blank">Description</a> |
| PACE OCI Level-2 Regional Inherent Optical Properties Data | PACE_OCI_L2_IOP | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3620139828-OB_CLOUD.html" target="_blank">Description</a> |
| PACE OCI Level-2 Regional Apparent Optical Properties Data | PACE_OCI_L2_AOP | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3620139598-OB_CLOUD.html" target="_blank">Description</a> |
| PACE OCI Level-4 Regional Mapped Multi-Ordination ANAlysis (MOANA) Data | PACE_OCI_L4M_MOANA | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3752338477-OB_CLOUD.html" target="_blank">Description</a> |

### Learning outcomes

At the end of this notebook you will know;
* how to search and download data from the NASA Earthdata portal Data Store using the <font color="#138D75">**earthaccess**</font> toolkit
* How to refine your <font color="#138D75">**searches**</font> for NASA PACE OCI marine products
* How to refine your searches for temporal, spatial, orbital and flag coverage parameters
* How to <font color="#138D75">**download**</font> full products

### Outline

The Earthdata portal provides access to all NASA PACE OCI data. Earthdata can be accessed via GUI and offers a series of APIs to facilitate programmatic interfaces. In this notebook, we will show you how to use these APIs to search for, filter and download PACE OCI products, by exploiting the `earthaccess` <a href="https://www.earthdata.nasa.gov/data/tools/earthaccess" target="_target">toolbox</a>.

<div class="alert alert-danger" role="alert">

This notebook offers only a small window into the capability of `earthaccess`. For a more comprehensive overview, we highly recommend that you review the **<a href="../submodules/pace-2025/book/presentations/notebooks/earthdata_cloud_access.ipynb" target="_blank">Earthdata Cloud Access</a>** notebook, included in this repository under the pace-2025 submodule.

</div>

<hr>

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python using the environment file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [1]:
import earthaccess            # a library that facilitates access to Earthdata products
import xarray as xr           # a library that supports the use of multi-dimensional arrays in Python
import numpy as np            # a library that provides support for array-based mathematics
import os                     # a library that allows us access to basic operating system commands like making directories
import datetime               # a library that allows us to work with dates and times

Next we will create a download directory to store the products we will download in this notebook.

In [2]:
download_dir = os.path.join(os.path.dirname(os.getcwd()), "Data", "Preprepared", "Day1", "Data_access", "OCI")
os.makedirs(download_dir, exist_ok=True)

<div class="alert alert-info" role="alert">

##  Downloading from the Earthdata Portal using the earthaccess toolbox

</div>

<div class="alert alert-block alert-success">

### Accessing Earthdata

To download NASA PACE OCI products from the Earthdata portal, we will use the `earthaccess` <a href="https://www.earthdata.nasa.gov/data/tools/earthaccess" target="_target">toolbox</a>. If you are working with the recommended Anaconda Python distribution and used the environment file included in this repository (environment.yml) to build this python environment (as detailed in the README), you will already have installed this. If not, you can install the toolkit using;

`conda install -c conda-forge earthaccess`

To download data, you need to provide credentials. To obtain these you should first register at for an <a href="https://urs.earthdata.nasa.gov/" target="_blank">EarthData account</a> and note your `username` and `password` for use below. If you don't already have a configured credentials file (~/.netrc), the line below will create it.
</div>

In [3]:
auth = earthaccess.login(persist=True)

Now we have authorised the toolbox, we can make requests from Earthdata. Lets start forming our search by setting the required instrument.

In [4]:
results = earthaccess.search_datasets(instrument="oci")

Lets now select the product we want from this intrument.

In [5]:
product = "PACE_OCI_L2_BGC"

We want to refine our search by time and space. We can do this by setting an extraction box and start and end times, as follows;

In [6]:
# space filter for matching products
west = 2.0
south = 54.0
east = 3.0
north = 55.0

# time filter for matching products
dtstart = datetime.datetime(2025, 8, 13, 0, 0)
dtend = datetime.datetime(2025, 8, 14, 0, 0)

Lets now use the information above to set our time span (`tspan`) and bbox spatial extent arguments. We can also define the minimum and maximum threshold for cloud coverage in our search results using the `clouds` argument.

In [7]:
tspan = (dtstart.strftime("%Y-%m-%d"), dtstart.strftime("%Y-%m-%d"))
bbox = (west, south, east, north)
clouds = (0, 100)

In [8]:
results = earthaccess.search_data(
        short_name=product,
        temporal=tspan,
        bounding_box=bbox,
        cloud_cover=clouds)
paths = earthaccess.download(results, local_path=download_dir)

/opt/miniconda3/envs/cmts_ocean_colour_applications/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/opt/miniconda3/envs/cmts_ocean_colour_applications/lib/python3.12/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2 [00:00<?, ?it/s]

If we are downloading level-3 or level-4 gridded products, we need to specify a timebase for the matching granules. We can do this as follows;

In [9]:
product_L4 = "PACE_OCI_L4M_MOANA"

results = earthaccess.search_data(
    short_name=product_L4,
    temporal=tspan,
    bounding_box=bbox,
    granule_name="*.DAY.*.0p1deg.*",
)
paths = earthaccess.download(results, local_path=download_dir)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

<div class="alert alert-success" role="alert">

## What next?

You should be able to adapt the methods above to search the Earthdata portal for the NASA PACE OCI data you require for this course, using your project ROI and time bounds a as input. If you need more information on using `earthaccess` to interface with the Earthdata you can find a comprehensive overview in the `earthaccess` <a href="https://www.earthdata.nasa.gov/data/tools/earthaccess" target="_target">online manual</a>.

</div>

<hr>
<a href="../Index.ipynb">Index</a> | <a href="./Access_OLCI_EUMETSAT_Data_Store.ipynb">Accessing Sentinel-3 OLCI data through the EUMETSAT Data Store</a> | <a href="./Access_MSI_CDSE.ipynb">Accessing Sentinel-2 MSI data via the CDSE</a> | <a href="./Access_multisensor_CMEMS_Data_Store.ipynb">Accessing multisensor data through the CMEMS Data Store</a>
<hr>
<a href="https://gitlab.eumetsat.int/eumetlab/ocean">View on GitLab</a> | <a href="https://training.eumetsat.int/">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int>Contact helpdesk for support </a> | <a href=mailto:.training@eumetsat.int>Contact our training team to collaborate on and reuse this material</a></span></p>